# Notebook 01 — Dados: Aquisição, Conversão e Exploração

Este notebook documenta o **Milestone 1** do projeto:
- De onde veio o dataset e por que foi escolhido
- Como os dados foram convertidos de 2D JPG → 3D NIfTI
- Validação do loader
- Exploração visual: fatias axial/coronal/sagital
- Distribuição de valores HU
- Demonstração de carregamento DICOM com pydicom

## 1. Dataset

**Fonte:** [Computed Tomography Images for Intracranial Hemorrhage Detection and Segmentation](https://www.kaggle.com/datasets/vbookshelf/computed-tomography-ct-images) (Kaggle, Hssayeni et al. 2019)

**Por que esse dataset:**
- 82 CTs de crânio com máscaras de segmentação anotadas por dois radiologistas
- 36 com hemorragia intracraniana, 46 sem → dataset razoavelmente balanceado
- Download direto via Kaggle CLI, sem registro extra
- Tamanho gerenciável (~90 MB)

**Download:**
```bash
kaggle datasets download -d vbookshelf/computed-tomography-ct-images \
  --path /tmp/kaggle_test --unzip
```

**Estrutura original do dataset:**
```
Patients_CT/
├── 049/brain/1.jpg, 2.jpg, ...          ← fatias CT (brain window)
├── 049/brain/6_HGE_Seg.jpg, ...         ← máscaras das fatias com hemorragia
├── 050/brain/...
...
hemorrhage_diagnosis.csv                  ← labels por fatia (tipo de hemorragia)
```

## 2. Conversão: 2D JPG → 3D NIfTI

O dataset original usa **fatias 2D em JPG** — um formato comum quando dados são exportados de sistemas hospitalares (PACS) sem acesso ao DICOM original.

Nosso modelo é uma **3D U-Net** que precisa de volumes tridimensionais. Por isso, convertemos:

```
049/brain/1.jpg  ┐
049/brain/2.jpg  ├─ empilha → 049.nii.gz  (33 × 650 × 650)
049/brain/3.jpg  ┘              049_mask.nii.gz
```

**Decisão de design:** As imagens JPG já foram exportadas com a janela cerebral aplicada (pixels 0–255 representando o intervalo [-5, 75] HU). Fizemos o mapeamento reverso para restaurar os valores em HU antes de salvar como NIfTI — assim o pipeline de pré-processamento funciona sem modificações.

```python
# Reverse-map: pixel [0,255] → HU [-5, 75]
hu = pixel * (75 - (-5)) / 255.0 + (-5)
```

**Script de conversão:** `scripts/convert_jpg_to_nifti.py`
```bash
python scripts/convert_jpg_to_nifti.py \
    --dataset-dir /tmp/kaggle_test/computed-tomography-images-... \
    --out-dir data/raw
```

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.data.loader import build_index, load_nifti, train_val_split
from src.visualization.plots import show_slices

DATA_ROOT = '../data/raw'
print('Imports OK')

## 3. Validação do loader

In [ ]:
records = build_index(DATA_ROOT)
train_records, val_records = train_val_split(records, val_fraction=0.2)

print(f'Total de estudos : {len(records)}')
print(f'Treino           : {len(train_records)}')
print(f'Validação        : {len(val_records)}')
print()
print('Exemplo de registro:')
print(' volume:', records[0]['image'])
print(' máscara:', records[0]['mask'])

In [ ]:
# Carregar um volume e sua máscara
volume, spacing = load_nifti(records[0]['image'])
mask, _         = load_nifti(records[0]['mask'])
mask = (mask > 0.5).astype(np.uint8)

print('Shape do volume  :', volume.shape, '  → (fatias, altura, largura)')
print('Espaçamento (mm) :', spacing, ' → [5mm entre fatias, 1mm em-plano]')
print('Intervalo HU     :', volume.min().round(1), '→', volume.max().round(1))
print()
print('Shape da máscara :', mask.shape)
print('Valores únicos   :', np.unique(mask).tolist(), ' → binário {0, 1}')
print('Voxels de lesão  :', mask.sum(), f'({100 * mask.mean():.3f}% do volume)')

In [ ]:
# Distribuição com/sem lesão no dataset
n_com_lesao = sum(1 for r in records if (load_nifti(r['mask'])[0] > 0.5).any())
n_sem_lesao = len(records) - n_com_lesao

print(f'Com hemorragia : {n_com_lesao}')
print(f'Sem hemorragia : {n_sem_lesao}')

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Com hemorragia', 'Sem hemorragia'], [n_com_lesao, n_sem_lesao],
       color=['#e74c3c', '#95a5a6'])
ax.set_ylabel('Pacientes')
ax.set_title('Distribuição do dataset')
for i, v in enumerate([n_com_lesao, n_sem_lesao]):
    ax.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Visualização: fatias axial, coronal e sagital

Um CT de crânio pode ser visto em três planos:
- **Axial**: vista de cima (como se olhasse de cima para baixo na cabeça)
- **Coronal**: vista de frente (como se olhasse de frente para trás)
- **Sagital**: vista lateral (como se olhasse da orelha)

In [ ]:
# Paciente sem lesão
vol_sem, _ = load_nifti(next(r for r in records
                              if not (load_nifti(r['mask'])[0] > 0.5).any())['image'])
show_slices(vol_sem, title='Paciente sem hemorragia — fatias centrais')

In [ ]:
# Paciente com lesão + overlay da máscara
vol_com, _ = load_nifti(next(r for r in records
                              if (load_nifti(r['mask'])[0] > 0.5).any())['image'])
msk_com, _ = load_nifti(next(r for r in records
                              if (load_nifti(r['mask'])[0] > 0.5).any())['mask'])
msk_com = (msk_com > 0.5).astype(np.uint8)

show_slices(vol_com, mask=msk_com, title='Paciente com hemorragia — máscara em vermelho')

## 5. Distribuição de valores HU

A **janela cerebral** [-5, 75] HU captura os tecidos relevantes:

| Tecido | HU aproximado |
|---|---|
| LCR (líquido cefalorraquidiano) | 0–10 |
| Substância branca | 25–30 |
| Substância cinzenta | 35–40 |
| Sangue / hemorragia | 50–80 |

Osso (>700 HU) e ar (~-1000 HU) foram descartados na conversão — não contribuem para detecção de lesão.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histograma do volume
axes[0].hist(vol_com.flatten(), bins=80, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Hounsfield Units (HU)')
axes[0].set_ylabel('Contagem de voxels')
axes[0].set_title('Distribuição HU — paciente com hemorragia')
axes[0].axvline(65, color='red', linestyle='--', alpha=0.7, label='Hemorragia ~65 HU')
axes[0].axvline(35, color='orange', linestyle='--', alpha=0.7, label='Córtex ~35 HU')
axes[0].legend()

# Comparação: voxels de lesão vs tecido normal
lesion_hu  = vol_com[msk_com == 1].flatten()
normal_hu  = vol_com[msk_com == 0].flatten()
axes[1].hist(normal_hu, bins=60, color='steelblue', alpha=0.6, label=f'Tecido normal (n={len(normal_hu):,})', density=True)
axes[1].hist(lesion_hu, bins=60, color='red',       alpha=0.6, label=f'Lesão (n={len(lesion_hu):,})', density=True)
axes[1].set_xlabel('Hounsfield Units (HU)')
axes[1].set_ylabel('Densidade')
axes[1].set_title('HU: lesão vs tecido normal')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'HU médio na lesão  : {lesion_hu.mean():.1f}')
print(f'HU médio no tecido : {normal_hu.mean():.1f}')

## 6. Demonstração DICOM com pydicom

O pipeline principal usa NIfTI, mas **DICOM é o formato real de hospitais**.
O módulo `dicom_demo.py` demonstra como carregar e processar um arquivo DICOM:
- Leitura de metadados clínicos (ID do paciente, data, equipamento)
- Conversão de pixel values → Hounsfield Units via `RescaleSlope` e `RescaleIntercept`
- Empilhamento de série (múltiplos `.dcm` → volume 3D)

Usamos um arquivo de teste que já vem instalado com o `pydicom`.

In [ ]:
from src.data.dicom_demo import demo_with_pydicom_testfile

hu_slice, meta = demo_with_pydicom_testfile()

print('Metadados DICOM:')
for k, v in meta.items():
    print(f'  {k:25s}: {v}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].imshow(hu_slice, cmap='gray')
axes[0].set_title('Fatia CT carregada via pydicom (valores em HU)')
axes[0].axis('off')

axes[1].hist(hu_slice.flatten(), bins=80, color='teal', alpha=0.8)
axes[1].set_xlabel('Hounsfield Units')
axes[1].set_title('Distribuição HU — arquivo DICOM de teste')

plt.tight_layout()
plt.show()

## Resumo do Milestone 1

| Etapa | Status |
|---|---|
| Ambiente virtual Python 3.13 | ✅ |
| Instalação de dependências | ✅ |
| Download do dataset (Kaggle) | ✅ |
| Conversão JPG 2D → NIfTI 3D | ✅ |
| Loader NIfTI funcionando | ✅ |
| Demonstração DICOM (pydicom) | ✅ |

**Próximo passo:** Notebook 02 — Pipeline de Pré-processamento